# Task
Consigne 2 : Modélisation en NLP

Here is all the data you need:
"submission.csv"

## Data loading

### Subtask:
Load the data from the "submission.csv" file into a pandas DataFrame.


**Reasoning**:
Load the data from the "submission.csv" file into a pandas DataFrame and display the first few rows.



In [1]:
import pandas as pd

try:
    df = pd.read_csv('submission.csv')
    display(df.head())
except FileNotFoundError:
    print("Error: 'submission.csv' not found.")
    df = None
except pd.errors.ParserError:
    print("Error: Could not parse 'submission.csv'. Check the file format.")
    df = None
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    df = None

,textID,selected_text
0,f87dea47db,Last session of the day http://twitpic.com/67ezh
1,96d74cb729,exciting
2,eee518ae67,Recession
3,01082688c6,happy bday!
4,33987a8ee5,I like it!!


## Data exploration

### Subtask:
Explore the loaded data to understand its structure and identify key features.


**Reasoning**:
Examine the shape, data types, missing values, descriptive statistics, and identify key features of the DataFrame `df`.



In [2]:
# Examine the shape of the DataFrame
print("Shape of the DataFrame:", df.shape)

# Display data types of each column
print("\nData types of each column:\n", df.dtypes)

# Check for missing values
print("\nMissing values:\n", df.isnull().sum())

# Analyze the distribution of the target variable (selected_text) - text length
df['selected_text_length'] = df['selected_text'].str.len()
print("\nDescriptive statistics of selected_text length:\n", df['selected_text_length'].describe())

# Identify relevant features for NLP modeling
print("\nRelevant features for NLP modeling task: 'selected_text'")

Shape of the DataFrame: (3534, 2)

Data types of each column:
 textID           object
selected_text    object
dtype: object

Missing values:
 textID           0
selected_text    0
dtype: int64

Descriptive statistics of selected_text length:
 count    3534.000000
mean       42.120260
std        38.842479
min         2.000000
25%         7.000000
50%        30.000000
75%        67.750000
max       138.000000
Name: selected_text_length, dtype: float64

Relevant features for NLP modeling task: 'selected_text'


## Data preparation

### Subtask:
Preprocess the text data in the 'selected_text' column of the DataFrame `df`.


**Reasoning**:
Preprocess the text data in the 'selected_text' column by cleaning, tokenizing, and optionally stemming or lemmatizing the text.  Create a new column 'preprocessed_text' to store the results.



In [4]:
import nltk
nltk.download('punkt_tab')

import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

def preprocess_text(text):
    # Clean the text
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    text = text.lower()  # Convert to lowercase

    # Tokenize the text
    tokens = word_tokenize(text)

    # Remove stop words
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]

    # Stem the tokens
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(token) for token in tokens]

    return " ".join(tokens)

df['preprocessed_text'] = df['selected_text'].apply(preprocess_text)
display(df.head())

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,textID,selected_text,selected_text_length,preprocessed_text
0,f87dea47db,Last session of the day http://twitpic.com/67ezh,49,last session day httptwitpiccom67ezh
1,96d74cb729,exciting,8,excit
2,eee518ae67,Recession,9,recess
3,01082688c6,happy bday!,12,happi bday
4,33987a8ee5,I like it!!,11,like


## Feature engineering

### Subtask:
Create numerical features from the preprocessed text data using TF-IDF.


**Reasoning**:
Create numerical features from the preprocessed text data using TF-IDF, then convert the sparse matrix to a dense array and create a new DataFrame.



In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Initialize TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# Fit and transform the preprocessed text
tfidf_matrix = vectorizer.fit_transform(df['preprocessed_text'])

# Convert to dense array
tfidf_array = tfidf_matrix.toarray()

# Create a new DataFrame from the TF-IDF array
tfidf_df = pd.DataFrame(tfidf_array, columns=vectorizer.get_feature_names_out())

display(tfidf_df.head())

,000 httpplurkcompwxj54,010,010 050,050,050 slowli,09,0930,0930 rang,10,10 167,...,yupz shower,zack,zack save,zero,zero contact,zimbabw,zombi,zombi your,zoo,zoo danwtmoon
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Model training

### Subtask:
Train a Logistic Regression model on the TF-IDF features.


**Reasoning**:
Train a Logistic Regression model on the TF-IDF features. This involves importing necessary libraries, instantiating the model, splitting the data, and training the model.



In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Instantiate the Logistic Regression model
logreg_model = LogisticRegression(solver='liblinear', max_iter=1000, random_state=42)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(tfidf_df, df['selected_text_length'], test_size=0.2, random_state=42)

# Train the Logistic Regression model
logreg_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

## Model evaluation

### Subtask:
Evaluate the trained Logistic Regression model's performance.


**Reasoning**:
Evaluate the trained Logistic Regression model using the test data and calculate the relevant metrics.



In [7]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Make predictions on the test set
y_pred = logreg_model.predict(X_test)

# Calculate evaluation metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

# Print the evaluation metrics
print(f"Mean Squared Error (MSE): {mse}")
print(f"R-squared (R2): {r2}")
print(f"Mean Absolute Error (MAE): {mae}")

# Discuss the results (This is a placeholder, replace with your analysis)
print("\nDiscussion:")
if r2 > 0.5:
    print("The R-squared value suggests a reasonably good fit.")
elif r2 > 0.3:
    print("The R-squared value suggests a moderate fit.")
else:
    print("The R-squared value indicates a poor fit.")

print(f"The MSE of {mse} and MAE of {mae} provide further insight into the model's predictive accuracy.")

Mean Squared Error (MSE): 2987.016973125884
R-squared (R2): -0.9335561982997587
Mean Absolute Error (MAE): 38.03536067892504

Discussion:
The R-squared value indicates a poor fit.
The MSE of 2987.016973125884 and MAE of 38.03536067892504 provide further insight into the model's predictive accuracy.


## Summary:

### 1. Q&A

The analysis aimed to predict the length of the `selected_text` using a Logistic Regression model.  How well did the model perform?

The model performed poorly.  The R-squared value was -0.93, indicating a worse-than-average fit.  The MSE was 2987.02, and the MAE was 38.04, further highlighting the model's inaccuracy.

### 2. Data Analysis Key Findings

* **Data Shape:** The dataset contains 3534 rows and 2 columns ('textID', 'selected_text').
* **Target Variable Analysis:** The average length of the `selected_text` is 42.12 characters, with a standard deviation of 38.84, minimum of 2, and maximum of 138.
* **Model Performance:**  The trained Logistic Regression model exhibits poor performance, with an R-squared of -0.93, MSE of 2987.02, and MAE of 38.04.  This indicates the model does not effectively predict the `selected_text` length.

### 3. Insights or Next Steps

* **Investigate Model Choice and Features:** The negative R-squared strongly suggests the chosen Logistic Regression model is inappropriate for predicting the length of text. Explore other regression models (e.g., linear regression, random forest regressor) or consider engineering different features.
* **Re-evaluate Feature Engineering:** The TF-IDF features might not be capturing the relevant information for predicting text length. Explore alternative feature engineering techniques, such as word embeddings or sentiment analysis, to better represent the text data.
